# Experiments & ablations

Use **`notebooks/EDA.ipynb`** for the full pipeline: environment setup, preprocessing, Model A training, Model B training, and artifact checks.


- Compare metrics across runs (e.g. after changing hyperparameters).
- Log tables or plots for ensemble weight sweeps, `top_k` changes, or capped training sizes.


## 1. Project root (Colab or local)

Reuse the same idea as `EDA.ipynb`: run from `race_rc_project` or let the cell locate it under `/content`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()


def find_project_root(start: Path) -> Path | None:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / "requirements.txt").exists() and (p / "src").exists():
            return p
    for p in (
        Path("/content/race_rc_project"),
        Path("/content/Intelligent-Reading-Comprehension-and-Quiz-Generation-System-using-Machine-Learning-Neural-Networks/race_rc_project"),
    ):
        if (p / "requirements.txt").exists() and (p / "src").exists():
            return p
    base = Path("/content")
    if base.exists():
        for p in base.rglob("race_rc_project"):
            if (p / "requirements.txt").exists() and (p / "src").exists():
                return p
    return None


project_root = find_project_root(ROOT)
if project_root is None:
    raise FileNotFoundError("Clone the repo and open this notebook from race_rc_project (see EDA.ipynb).")
if Path.cwd().resolve() != project_root.resolve():
    %cd {project_root}
print("Project root:", Path.cwd())
print("Python:", sys.version.split()[0])

## 2. Snapshot: Model A / Model B metrics

After any training run, compare validation vs test without opening JSON by hand.

In [ ]:
import json
from pathlib import Path

def show_metrics(path: Path, title: str) -> None:
    if not path.exists():
        print(f"[missing] {title}: {path}")
        return
    data = json.loads(path.read_text(encoding="utf-8"))
    print("===", title, "===")
    print("config:", json.dumps(data.get("config", {}), indent=2))
    for split in ("validation", "test"):
        if split not in data:
            continue
        print(f"\n-- {split} --")
        block = data[split]
        if isinstance(block, dict) and any(k in block for k in ("distractor_generation", "hint_generation", "question_generation")):
            for key in ("question_generation", "distractor_generation", "hint_generation"):
                if key in block:
                    print(key + ":", json.dumps(block[key], indent=2))
        else:
            print(json.dumps(block, indent=2))


show_metrics(Path("models/model_a/traditional/metrics_summary.json"), "Model A")
print()
show_metrics(Path("models/model_b/traditional/metrics_summary.json"), "Model B")
print()
show_metrics(Path("models/model_bert/metrics_summary.json"), "BERT rerank")

## 3. BERT vs traditional baselines (side-by-side)

`models/comparison_summary.json` is written by `python -m src.model_bert_train`
and contains a per-task comparison between the classical pipelines (Model A,
Model B) and the BERT rerank baseline. Each row is a metric; each column is
`{model}.{split}`.

In [ ]:
import json
from pathlib import Path

import pandas as pd

_METRIC_COLS = ("bleu_mean", "rouge1_f_mean", "rouge2_f_mean", "rougeL_f_mean", "meteor_mean")


def build_task_table(task_block: dict) -> pd.DataFrame:
    """Convert one task entry from comparison_summary.json into a tidy DataFrame."""
    rows = {m: {} for m in _METRIC_COLS + ("n_examples",)}
    for split, models in task_block.items():
        for model_name, metrics in (models or {}).items():
            if not metrics:
                continue
            col = f"{model_name}.{split}"
            for m in _METRIC_COLS + ("n_examples",):
                rows[m][col] = metrics.get(m)
    df = pd.DataFrame(rows).T
    df.index.name = "metric"
    return df


comp_path = Path("models/comparison_summary.json")
if not comp_path.exists():
    print("Run `python -m src.model_bert_train` first to produce", comp_path)
else:
    comp = json.loads(comp_path.read_text(encoding="utf-8"))
    for task, block in comp["tasks"].items():
        print("===", task, "===")
        tbl = build_task_table(block)
        try:
            display(tbl.style.format("{:.4f}", na_rep="—", subset=pd.IndexSlice[list(_METRIC_COLS), :]))
        except NameError:
            print(tbl.to_string())
        print()

## 4. Run the BERT baseline on Colab (recommended for full val+test)

`bert-base-uncased` is slow on a laptop CPU (≈ 1–3 h on full val/test). On a
free Colab T4 / L4 GPU the same run finishes in **5–10 min**. The cells below
mirror what `python -m src.model_bert_train` does locally — they install
torch + transformers, locate the repo (Drive / clone), and run the trainer
with `--device cuda --fp16`.

In [ ]:
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    print("Colab detected — make sure the runtime is set to GPU (Runtime → Change runtime type → T4 GPU).")
    !pip -q install "transformers>=4.41,<5" "rouge-score>=0.1.2,<1" nltk pyarrow joblib
    !pip -q install torch  # Colab already ships torch with CUDA; this is a no-op upgrade
else:
    print("Not on Colab — skip this cell. Use `pip install -r requirements.txt` locally.")

In [ ]:
import torch
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

device_flag = "cuda" if torch.cuda.is_available() else "cpu"
fp16_flag = "--fp16" if torch.cuda.is_available() else ""
batch = 64 if torch.cuda.is_available() else 32

cmd = (
    f"python -m src.model_bert_train "
    f"--processed-dir data/processed "
    f"--output-dir models/model_bert "
    f"--model-name bert-base-uncased "
    f"--max-val-mcq 2011 --max-test-mcq 5027 "
    f"--device {device_flag} --batch-size {batch} --max-length 128 {fp16_flag}"
).strip()
print(cmd)
assert Path("src/model_bert_train.py").exists(), "Run section 1 first to cd into race_rc_project."
!{cmd}

After the run, copy `models/model_bert/` and `models/comparison_summary.json`
back to your local repo (e.g. via Drive or `files.download`). Then re-run
section 2 and section 3 above to render the comparison tables.

## 3. Example ablations (run in separate runs, then re-run section 2)

Ideas to document in your report (adjust flags to match your CLI — see `README.md`):

- **Ensemble balance**: train Model A or B with a different supervised weight (e.g. `0.3` vs `0.7`) and compare BLEU/ROUGE/METEOR.
- **Candidate breadth**: change how many sentences or terms are considered (if exposed in `TrainConfig` / argparse).
- **Smaller data**: cap `max_train_mcq` for a fast sanity curve vs full data.

Keep one row per run: date, commit hash, command, and path to saved `metrics_summary.json` (or copy the JSON into this notebook output).